# 04 - Lakebase round trip
Synced table -> review queue (routing rule) -> reviewer override -> clause_reviews -> synced back to Delta gold_reviewed_clauses. Postgres tables are read through their Delta mirrors written by the sync job.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from dbx import sql, ws, CATALOG
print('workspace:', ws().config.host, '| catalog:', CATALOG, '| user:', ws().current_user.me().user_name)

workspace: https://fevm-serverless-stable-oj7eie.cloud.databricks.com | catalog: serverless_stable_oj7eie_catalog | user: erica.yuen@databricks.com


In [2]:
st = ws().database.get_synced_database_table(CATALOG + '.serving.review_queue_synced')
print(st.name, st.data_synchronization_status.detailed_state, st.spec.scheduling_policy, st.spec.source_table_full_name)
inst = ws().database.get_database_instance('matter-intel-lakebase'); print(inst.name, inst.state, inst.capacity)

serverless_stable_oj7eie_catalog.serving.review_queue_synced SyncedTableState.SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE SyncedTableSchedulingPolicy.SNAPSHOT serverless_stable_oj7eie_catalog.serving.review_queue_source
matter-intel-lakebase DatabaseInstanceState.AVAILABLE CU_1


In [3]:
sql("SELECT initial_status, severity, count(*) n, round(min(confidence),2) min_conf FROM ${catalog}.serving.review_queue_source GROUP BY ALL ORDER BY 1,2")

,initial_status,severity,n,min_conf
0,auto_accepted,low,32,0.9
1,auto_accepted,medium,14,0.95
2,auto_accepted,none,90,0.9
3,pending,high,14,0.85
4,pending,low,1,0.65
5,pending,medium,2,0.75
6,pending,none,13,0.3


In [4]:
sql("SELECT status, count(*) n FROM ${catalog}.serving.pg_review_queue GROUP BY 1")

,status,n
0,auto_accepted,135
1,pending,27
2,overridden,2
3,approved,2


In [5]:
rt = sql("SELECT run_id, step_no, step, ts, payload FROM ${catalog}.ml.roundtrip_evidence ORDER BY ts DESC LIMIT 5").sort_values('step_no')
for r in rt.itertuples(): print(f'--- {r.step_no} {r.step} @ {r.ts}'); print(r.payload[:1200])

--- 1 1_review_queue_before @ 2026-11-18T01:31:21.138Z
{
 "clause_id": "74c1da82ee8ce1ca111ae11240f2c93fb9e365ea047da52da973aa1f88c95048",
 "doc_id": "661d35cc01bcb542a595c245b317e8a4e508dfa62c32cee0125afae43d5d08f2",
 "doc_title": "RESELLER AGREEMENT",
 "contract_type": "Distribution",
 "matter_team": "Team Denali",
 "clause_type": "liability_cap",
 "current_value": "Reseller not liable to Company except for willful misconduct or gross negligence",
 "confidence": 0.85,
 "severity": "none",
 "status": "pending",
 "updated_at": "2026-11-18 01:25:07.934395+00:00"
}
--- 2 2_override_action @ 2026-11-18T01:31:21.171Z
{
 "request": {
  "function": "lakebase.override",
  "clause_id": "74c1da82ee8ce1ca111ae11240f2c93fb9e365ea047da52da973aa1f88c95048",
  "reviewer": "erica.yuen@databricks.com",
  "new_value": "USD 1,000,000 aggregate cap (reviewer override)",
  "comment": "Round-trip demo: cap confirmed with counterparty at USD 1M"
 },
 "response": {
  "review_id": 1,
  "reviewed_at": "2026-11

In [6]:
sql("SELECT * FROM ${catalog}.serving.pg_clause_reviews ORDER BY review_id")

,review_id,clause_id,reviewer,action,old_value,new_value,comment,reviewed_at
0,1,74c1da82ee8ce1ca111ae11240f2c93fb9e365ea047da52da973aa1f88c95048,erica.yuen@databricks.com,override,Reseller not liable to Company except for willful misconduct or gross negligence,"USD 1,000,000 aggregate cap (reviewer override)",Round-trip demo: cap confirmed with counterparty at USD 1M,2026-11-18T01:31:21.140
1,32,be3ccc77dc240cdf2cdfc31a3516bd6d84ba64c8f260f47cc05be36cf4ec454b,erica.yuen@databricks.com,approve,"Company prohibited from marketing, promoting, selling, distributing Company Products or similar products in Territor...","Company prohibited from marketing, promoting, selling, distributing Company Products or similar products in Territor...",approved in app,2026-11-18T02:09:24.306
2,33,8e9f51f852f80feee4689cf85c0019d93b417862c431fbb1c349e263ba1a4758,erica.yuen@databricks.com,approve,Virginia,Virginia,approved in app,2026-11-18T02:10:03.581
3,34,23074043af6c7466be9935a3bf7eabde66b85dddea0ecd725771127d53b25071,erica.yuen@databricks.com,override,2018-04-07,2018-04-07,Expiration date is not explicit here. None listed,2026-11-18T02:10:57.497


In [7]:
sql("SELECT clause_id, clause_type, extracted_value, reviewed_value, review_action, reviewer, reviewed_at, synced_at FROM ${catalog}.curated.gold_reviewed_clauses")

,clause_id,clause_type,extracted_value,reviewed_value,review_action,reviewer,reviewed_at,synced_at
0,23074043af6c7466be9935a3bf7eabde66b85dddea0ecd725771127d53b25071,expiration_date,2018-04-07,2018-04-07,override,erica.yuen@databricks.com,2026-11-18T02:10:57.497Z,2026-11-18T02:12:08.463Z
1,be3ccc77dc240cdf2cdfc31a3516bd6d84ba64c8f260f47cc05be36cf4ec454b,non_compete,"Company prohibited from marketing, promoting, selling, distributing Company Products or similar products in Territor...","Company prohibited from marketing, promoting, selling, distributing Company Products or similar products in Territor...",approve,erica.yuen@databricks.com,2026-11-18T02:09:24.306Z,2026-11-18T02:12:08.463Z
2,8e9f51f852f80feee4689cf85c0019d93b417862c431fbb1c349e263ba1a4758,governing_law,Commonwealth of Virginia,Virginia,approve,erica.yuen@databricks.com,2026-11-18T02:10:03.581Z,2026-11-18T02:12:08.463Z
3,74c1da82ee8ce1ca111ae11240f2c93fb9e365ea047da52da973aa1f88c95048,liability_cap,Reseller not liable to Company except for willful misconduct or gross negligence,"USD 1,000,000 aggregate cap (reviewer override)",override,erica.yuen@databricks.com,2026-11-18T01:31:21.140Z,2026-11-18T01:31:50.277Z


In [8]:
sql("SELECT actor, action, count(*) n, round(percentile(duration_ms, 0.95),1) p95_ms FROM ${catalog}.serving.pg_audit_log GROUP BY ALL ORDER BY 1,2")

,actor,action,n,p95_ms
0,erica.yuen@databricks.com,approve,2,18.6
1,erica.yuen@databricks.com,open_document,4,NaN
2,erica.yuen@databricks.com,override,2,22.0
3,erica.yuen@databricks.com,view_page,9,NaN
